In [ ]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset
from utils.hub_card import push_dataset_card
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [ ]:
# Pre-arrival medication reconciliation from ed.medrecon.
# Medications the patient was already taking before the ED visit.
# Treated as static/baseline state — not time-varying actions.
# Multiple rows per visit: one per drug per ETC therapeutic class group.
# etcdescription groups drugs by class (e.g. 'Beta Blockers', 'ACE Inhibitors').
# Deduplication applied: same patient + stay + drug identity collapsed to one row.

query_medrecon = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id
  FROM `ads-599-final.rl_project.cohort_base`
)
SELECT
  m.stay_id       AS ed_stay_id,
  m.subject_id,
  m.charttime,
  m.name,
  m.gsn,
  m.ndc,
  m.etccode,
  m.etcdescription
FROM `physionet-data.mimiciv_ed.medrecon` m
INNER JOIN cohort_subjects cs ON m.stay_id = cs.ed_stay_id
ORDER BY m.stay_id, m.name
"""

print("Running medrecon query...")
df_medrecon = client.query(query_medrecon).to_dataframe()
print(f"Shape before dedup: {df_medrecon.shape}")
print(f"Unique ED visits: {df_medrecon['ed_stay_id'].nunique():,}")
df_medrecon.head()

In [ ]:
# Deduplication: same patient + stay + drug identity can appear multiple times
# due to multiple GSN ontology group mappings. Collapse to one row per drug per visit.
dupes = df_medrecon.duplicated(subset=['subject_id', 'ed_stay_id', 'name', 'gsn', 'ndc', 'etccode']).sum()
print(f"Duplicate rows removed: {dupes:,}")
df_medrecon.drop_duplicates(subset=['subject_id', 'ed_stay_id', 'name', 'gsn', 'ndc', 'etccode'], inplace=True, ignore_index=True)
print(f"Shape after dedup: {df_medrecon.shape}")

In [ ]:
DESCRIPTION = (
    "Medication reconciliation records from ed.medrecon — medications the patient was taking "
    "prior to ED arrival. One row per medication per ED visit. "
    "Represents pre-arrival medication state, not actions taken during the visit."
)

ds = Dataset.from_pandas(df_medrecon, split='medrecon')
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="medrecon", data_dir="pre_arrival_meds")
push_dataset_card("ADS599-Capstone/raw_data", config_name="medrecon", split="medrecon", data_dir="pre_arrival_meds", description=DESCRIPTION)
print("Pushed medrecon to HuggingFace Hub.")